<a href="https://colab.research.google.com/github/josephzl04/image-data-generation/blob/main/notebooks/dissertation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate pillow safetensors controlnet-aux diffusers

In [ ]:
import io
import gc
import time
from pathlib import Path

import torch
import numpy as np
import cv2

from PIL import Image
from google.colab import files
from IPython.display import display

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

In [ ]:
repo_dir = Path("/content/image-data-generation")
output_dir = repo_dir / "results" / "main_pipeline_demo"
output_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Output folder:", output_dir)

In [ ]:
# Source image is uploaded, converted to RGB format and saved.
print("Please upload a road-scene image")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No input image was uploaded.")

image_name = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[image_name])).convert("RGB")

display(image)
print(f"Loaded image: {image_name}")

In [ ]:
# BLIP2 to generate caption
blip_dtype = torch.float16 if device == "cuda" else torch.float32

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=blip_dtype,
    device_map="auto",
)

inputs = processor(image, return_tensors="pt").to(blip_model.device)
with torch.no_grad():
    ids = blip_model.generate(**inputs, max_length=40)
caption = processor.decode(ids[0], skip_special_tokens=True)

if not caption or len(caption.strip()) == 0:
    raise ValueError("BLIP-2 did not generate a valid caption.")

print("\n Generated BLIP-2 Caption:", caption)

del blip_model
del processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# MiDaS Depth Map Generation
midas_device = torch.device(device)

midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

img_np = np.array(image)
input_batch = transform(img_np).to(midas_device)

with torch.no_grad():
    prediction = midas_model(input_batch)
    prediction = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=img_np.shape[:2],
        mode="bicubic",
        align_corners=False
    ).squeeze(0).squeeze(0)

depth = prediction.detach().cpu().numpy()

depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
depth_vis = depth_vis.astype(np.uint8)

depth_gray = Image.fromarray(depth_vis).convert("RGB")

depth_gray_path = output_dir / "depth_gray.png"
depth_gray.save(depth_gray_path)

if not depth_gray_path.exists():
    raise FileNotFoundError("Depth map was not saved.")

display(depth_gray)
print("Depth map saved to:", depth_gray_path)
print("Depth map mode:", depth_gray.mode)
print("Depth map size:", depth_gray.size)

del midas_model
del midas_transforms
del transform
gc.collect()
torch.cuda.empty_cache()

In [ ]:
init_image = image.copy().convert("RGB")
depth_image = Image.open(depth_gray_path).convert("RGB")

print("Upload conditional image for IP-Adapter")
uploaded_conditional = files.upload()

if len(uploaded_conditional) == 0:
    raise ValueError("No conditional image was uploaded.")

cond_name = list(uploaded_conditional.keys())[0]
ip_image = Image.open(io.BytesIO(uploaded_conditional[cond_name])).convert("RGB")

display(ip_image)
print("Loaded conditional image:", cond_name)

#resize image for consistency
init_image = init_image.resize((512,512))
depth_image = depth_image.resize((512,512))
ip_image= ip_image.resize((512,512))

#user inputs and blip2 output caption
user_prompt = input("Enter the transformation you want (e.g. 'snowy winter scene', 'cartoon style', 'oil painting'): ")

final_prompt = (
    f"Scene description: {caption}. "
    f"User prompt: {user_prompt}. "
    "Preserve original road layout, vehicles, object positions, and perspective."
)
negative_prompt = "blurry, low quality, warped perspective, distorted geometry, extra limbs, deformed face, bad anatomy, warped perspective, artifacts"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# load controlNet
print("Loading controlNet")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

# load stable diffusion
print("Loading stable diffusion")
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)


# load IP adapter
print("Loading IP adapter")
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

pipe.set_ip_adapter_scale(0.3)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers enabled")
except Exception as e:
    print("xFormers not available:", e)

generator = torch.Generator(device=device).manual_seed(42)

print("Generating final image")
start_time = time.time()
out = pipe(
    prompt=final_prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    control_image=depth_image,
    ip_adapter_image=ip_image,
    strength=0.85,
    guidance_scale=8.5,
    controlnet_conditioning_scale=0.8,
    num_inference_steps=40,
    generator=generator
).images[0]

runtime = time.time() - start_time
print(f"Runtime: {runtime:.2f} seconds")

translated_path = output_dir / "translated.png"
out.save(translated_path)

display(out)
print("Final output saved to:", translated_path)

metadata_path = output_dir / "metadata.txt"

with open(metadata_path, "w") as f:
    f.write(f"Input image: {image_name}\n")
    f.write(f"Reference image: {cond_name}\n")
    f.write(f"BLIP-2 caption: {caption}\n")
    f.write(f"User prompt: {user_prompt}\n")
    f.write(f"Final prompt: {final_prompt}\n")
    f.write(f"Negative prompt: {negative_prompt}\n")
    f.write("Seed: 42\n")
    f.write("Image size: 512x512\n")
    f.write("Strength: 0.85\n")
    f.write("Guidance scale: 8.5\n")
    f.write("ControlNet conditioning scale: 0.8\n")
    f.write("IP-Adapter scale: 0.3\n")
    f.write("Inference steps: 40\n")
    f.write(f"Runtime: {runtime:.2f} seconds\n")

print("Metadata saved to:", metadata_path)

expected_files = [
    "depth_gray.png",
    "translated.png",
    "metadata.txt"
]

missing_files = [
    file for file in expected_files
    if not (output_dir / file).exists()
]

if missing_files:
    print("Missing files:", missing_files)
else:
    print("All expected main pipeline files were saved.")

del pipe
del controlnet
gc.collect()
torch.cuda.empty_cache()

print("Pipeline complete")